# Qwen3.5-0.8B — baseline Colab tương thích Kaggle

Mặc định CPU; có thể chọn GPU ở title 2 để sinh nhanh hơn. Notebook không gọi Gemini, không train.
Title 4 kiểm tra baseline validation/seen/unseen: cache đủ thì bỏ qua nạp model và inference; cache thiếu chỉ sinh những mẫu thiếu.
Nếu cache Kaggle đủ cho cấu hình train/eval, có thể bỏ toàn bộ notebook baseline.

Ba notebook dùng cùng `DRIVE_BASE_DIR`, dữ liệu gốc `grpo/train_seen.jsonl`, `eval/test_seen.jsonl`, `eval/test_unseen.jsonl`.
Chia train/validation/test theo đúng thuật toán Kaggle, giữ đề bài test ngoài train/validation. Cả ba notebook cần ba file gốc để tái tạo cùng split; không thay bằng file test đã lọc.
Baseline/validation/eval mặc định **512 token**, thinking tắt; cache có dạng `key → {text, generated_tokens, truncated}` như Kaggle.

**Dùng baseline Kaggle:** tải `results/baseline_outputs.json` và `results/split_manifest.json` lên Drive.
Ở title 2, đặt `BASELINE_IMPORT_FILE` và `SPLIT_MANIFEST_IMPORT_FILE` tới hai file đó; dùng đúng dữ liệu, SEED, tỷ lệ split và budget đã chạy Kaggle.
Cache được đọc trực tiếp, không cần chuyển định dạng. File cache có thể đặt thẳng tại `<DRIVE_BASE_DIR>/qwen_grpo_v2/results/baseline_outputs.json`, khi đó `BASELINE_IMPORT_FILE` để trống.
Baseline mới lưu vào `qwen_grpo_v2/results`; cache Colab cũ dạng chuỗi/384 token không được coi là baseline của cấu hình mới.
Output/checkpoint được ghi trực tiếp lên Google Drive. Mỗi lần train/eval có thư mục riêng, giữ kết quả của lần trước.

Dùng ba dataset **soft-v2** đã sửa. Sau migration, manifest cũ không còn khớp; để `SPLIT_MANIFEST_IMPORT_FILE` trống hoặc import manifest mới. Cache baseline cũ vẫn dùng cho đúng hash prompt/split; title 4 sinh phần thiếu.


In [ ]:
# @title 1. Kết nối Drive và cài thư viện — hỗ trợ runtime CPU
import os
import sys
import subprocess
import importlib.metadata
import torch
from google.colab import drive

print('Baseline hỗ trợ CPU; không cần bật GPU trong Runtime.')
if torch.cuda.is_available():
    print(f'GPU có sẵn: {torch.cuda.get_device_name(0)}; chọn cpu/cuda ở title 2.')
if not os.path.isdir("/content/drive/MyDrive"):
    drive.mount("/content/drive")
if not os.path.isdir("/content/drive/MyDrive"):
    raise RuntimeError("Google Drive chưa mount thành công; dừng để tránh lưu nhầm vào ổ tạm Colab.")

# Pin các API dùng trong notebook; Qwen3.5 cần Transformers có hỗ trợ kiến trúc qwen3_5.
# Không thay PyTorch/CUDA do Colab cung cấp. Không cần vLLM, Gemma hay torchao.
loaded_before_install = [m for m in ("transformers", "accelerate") if m in sys.modules]
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "--upgrade",
    "transformers==5.3.0",
    "accelerate>=1.4.0,<2",
    "tqdm", "sentencepiece",
], check=True)
for package in ("transformers", "accelerate"):
    print(f"{package}: {importlib.metadata.version(package)}")
if loaded_before_install:
    raise RuntimeError("Thư viện đã được import trước khi cài đặt. Restart runtime, rồi chạy notebook từ cell 1.")
print("Môi trường Colab và Google Drive sẵn sàng.")

In [ ]:
# @title 2. Drive, dữ liệu và baseline tương thích Kaggle
import os
import json
from pathlib import Path

DRIVE_BASE_DIR = "/content/drive/MyDrive/vinfast/Tuan4_5_6"  # @param {type:"string"}
if not os.path.isdir('/content/drive/MyDrive'):
    raise RuntimeError('Chạy title 1 để mount Drive trước.')
if not Path(DRIVE_BASE_DIR).resolve().is_relative_to(Path('/content/drive/MyDrive')):
    raise ValueError('DRIVE_BASE_DIR phải nằm trong /content/drive/MyDrive/.')
RUN_DIR = os.path.join(DRIVE_BASE_DIR, 'qwen_grpo_v2')
RESULTS_DIR = os.path.join(RUN_DIR, 'results')
OUTPUT_DIR = os.path.join(RUN_DIR, 'checkpoints')
BEST_ADAPTER_DIR = os.path.join(RUN_DIR, 'best_adapter')
FINAL_ADAPTER_DIR = os.path.join(RUN_DIR, 'final_adapter')
os.makedirs(RESULTS_DIR, exist_ok=True)
TRAIN_FILE = os.path.join(DRIVE_BASE_DIR, 'grpo', 'train_seen.jsonl')
TEST_SEEN_FILE = os.path.join(DRIVE_BASE_DIR, 'eval', 'test_seen.jsonl')
TEST_UNSEEN_FILE = os.path.join(DRIVE_BASE_DIR, 'eval', 'test_unseen.jsonl')
MODEL_NAME = 'Qwen/Qwen3.5-0.8B'  # @param {type:"string"}
MAX_PROMPT_LENGTH = 2048  # @param {type:"integer"}
MAX_COMPLETION_LENGTH = 512  # @param {type:"integer"} # Cùng giá trị đã dùng sinh baseline Kaggle.
SEED = 42
VALIDATION_FRACTION = 0.10
TEST_FRACTION = 0.20
MAX_VALIDATION_SAMPLES = 32  # @param {type:"integer"}
MAX_EVAL_SAMPLES_SEEN = 100  # @param {type:"integer"} # 0 = toàn bộ held-out test.
MAX_EVAL_SAMPLES_UNSEEN = 100  # @param {type:"integer"}
BASELINE_IMPORT_FILE = ''  # @param {type:"string"} # Đường dẫn Drive tới baseline_outputs.json của Kaggle.
SPLIT_MANIFEST_IMPORT_FILE = ''  # @param {type:"string"} # split_manifest.json Kaggle; khuyến nghị để kiểm tra cùng dữ liệu/split.
EVALUATE_BENCHMARK_BASELINE = True  # @param {type:"boolean"}
if not 0 < VALIDATION_FRACTION < 1 or not 0 < TEST_FRACTION < 1 or VALIDATION_FRACTION + TEST_FRACTION >= 1:
    raise ValueError('Tỷ lệ validation/test không hợp lệ.')
if any(type(v) is not int or v < 0 for v in (MAX_EVAL_SAMPLES_SEEN, MAX_EVAL_SAMPLES_UNSEEN)) or MAX_VALIDATION_SAMPLES < 1:
    raise ValueError('Giới hạn eval phải >=0; MAX_VALIDATION_SAMPLES phải >=1.')
if MAX_PROMPT_LENGTH < 1 or MAX_COMPLETION_LENGTH < 1:
    raise ValueError('Ngân sách token phải >=1.')
print(f'Artifact lưu trên Drive: {RUN_DIR}')

MODEL_DEVICE = 'cpu'  # @param ["cpu", "cuda"] # GPU nhanh hơn khi cần sinh phần thiếu.
if MODEL_DEVICE not in ('cpu','cuda') or (MODEL_DEVICE=='cuda' and not torch.cuda.is_available()):
    raise ValueError('Chọn cpu hoặc bật GPU để chọn cuda.')


In [ ]:
# @title 3. Tokenizer và các hàm chung cho train / baseline / eval
# Chính sách offline được nhúng: không cần thêm file Python vào Kaggle Input.
"""Offline rules shared by dataset generation, packaging and notebook exports.

Approval means an explicit successful verification or documented offline curation;
it never means a failed API request was accepted. Human judge calibration remains
a separate measurement.
"""
import copy
import json

SOFT_POLICY_VERSION = "soft-v2"
SEEN_SOFT_CATEGORIES = (
    "semantic_completeness", "style_and_tone", "clarity_and_coherence",
    "conciseness_and_efficiency", "practical_examples",
)
UNSEEN_SOFT_CATEGORIES = (
    "target_audience", "reasoning_and_logic", "role_play_persona",
    "counterfactual_context", "safety_and_neutrality", "pedagogical_analogy",
    "critique_and_limitations",
)
ACCEPTED_SOFT_STATUSES = ("PASS", "REVIEWED_OFFLINE")
SOFT_CATEGORY_GUIDANCE = {
    "semantic_completeness": "Các ý bắt buộc phải được giải thích đúng và liên quan tới đề bài; chỉ nhắc từ khóa hoặc đưa thông tin sai không được tính là bao phủ nội dung.",
    "style_and_tone": "Dựa vào cách diễn đạt thực tế: rõ ràng, lịch sự, phù hợp phong cách được yêu cầu. Không đòi mô hình tự nhận là chuyên gia, không thưởng chỉ vì có thuật ngữ.",
    "clarity_and_coherence": "Các ý liên kết và không tự mâu thuẫn; người đọc theo dõi được trình tự. Không bắt buộc câu nối hay hình thức riêng nếu description không yêu cầu.",
    "conciseness_and_efficiency": "Không lặp lại cùng ý, không có lời dẫn/đệm không phục vụ nhiệm vụ. Độ dài cần để đáp ứng luật cứng không tự động bị coi là dài dòng.",
    "practical_examples": "Ví dụ hoặc công cụ phải cụ thể, liên quan và được áp dụng đúng vào nhiệm vụ; danh sách tên không liên quan không đủ đạt.",
    "target_audience": "Giải thích thuật ngữ cần thiết, dùng cách diễn đạt và độ sâu phù hợp đối tượng. Không cấm mọi thuật ngữ chuyên môn khi thuật ngữ được giải thích đúng.",
    "reasoning_and_logic": "Các bước hoặc quan hệ nhân quả phải đúng, nối được dữ kiện với kết luận; không chỉ liệt kê nhãn các bước hoặc thuật ngữ.",
    "role_play_persona": "Vai diễn thể hiện trong cách phân tích, lựa chọn thông tin và cách giao tiếp đúng nhiệm vụ. Không đủ đạt chỉ vì tự xưng vai diễn hay chèn từ chuyên môn.",
    "counterfactual_context": "Tuân thủ giả định được công bố xuyên suốt và suy luận nhất quán trong giả định đó. Không bác bỏ giả định vì khác thực tế và không tự thêm giả định khác.",
    "safety_and_neutrality": "Không bịa dữ kiện, quy kết hoặc khẳng định tuyệt đối thiếu căn cứ; thể hiện các góc nhìn mà description yêu cầu, không ép cân bằng giả giữa thông tin đúng và sai.",
    "pedagogical_analogy": "Ẩn dụ/ví dụ phải có sự tương ứng đúng với khái niệm và giúp giải thích; chỉ nhắc một hình ảnh so sánh không liên quan không đủ đạt.",
    "critique_and_limitations": "Nêu rủi ro hoặc điều kiện biên cụ thể gắn với giải pháp, giải thích vì sao có giới hạn; cảnh báo chung chung không đủ đạt.",
}


def soft_reward_spec(gate="all_hard"):
    if gate not in ("all_hard", "partial_hard"):
        raise ValueError("Soft gate must be all_hard or partial_hard")
    return {
        "version": SOFT_POLICY_VERSION,
        "aggregation_strategy": "hard_priority_gated",
        "hard_partial_weight": 0.70,
        "hard_all_pass_weight": 0.30,
        "soft_bonus_weight": 0.25,
        "soft_gate": gate,
        "maximum_reward": 1.25,
        "soft_aggregation": "mean_binary_per_criterion",
        "formula": "0.70 * hard_partial + 0.30 * all_hard_pass + 0.25 * soft_gate_pass * soft_score",
    }


def normalize_soft_constraints(values, require_verified=True, split=None):
    """Return accepted, validated criteria and rejected entries with reasons.

    Never let a flat JSON field bypass a rejected nested verification record;
    callers must merge that provenance before calling this function.
    """
    values = json.loads(values) if isinstance(values, str) else values
    if not isinstance(values, list) or not all(isinstance(v, dict) for v in values):
        raise ValueError("Soft constraints phải là list object JSON.")
    accepted, excluded, ids = [], [], set()
    allowed = set(SEEN_SOFT_CATEGORIES if split == "seen" else
                  UNSEEN_SOFT_CATEGORIES if split == "unseen" else
                  SEEN_SOFT_CATEGORIES + UNSEEN_SOFT_CATEGORIES)
    for original in values:
        item = copy.deepcopy(original)
        ident = item.get("id")
        if not isinstance(ident, str) or not ident.strip() or ident in ids:
            raise ValueError("Soft constraint id thiếu hoặc trùng.")
        ids.add(ident)
        status = item.get("verification_status", "UNVERIFIED")
        if status == "REJECT" or (require_verified and status not in ACCEPTED_SOFT_STATUSES):
            excluded.append({"constraint": item, "reason": f"verification_status={status}"})
            continue
        category = item.get("category") or ident.split(":", 1)[0]
        if category not in allowed:
            raise ValueError(f"Soft category không hợp lệ cho split={split}: {category}")
        description = item.get("description") or item.get("description_vi")
        rubric = item.get("rubric") or item.get("eval_rubric")
        if not isinstance(description, str) or not description.strip() or not isinstance(rubric, str) or not rubric.strip():
            raise ValueError(f"Description/rubric rỗng: {ident}")
        accepted.append({
            "id": ident, "category": category,
            "description": description.strip(), "rubric": rubric.strip(),
            "evaluation_guidance": item.get("evaluation_guidance") or SOFT_CATEGORY_GUIDANCE[category],
            "verification_status": status,
            "verified_by": item.get("verified_by", "unknown"),
            "policy_version": SOFT_POLICY_VERSION,
        })
    return accepted, excluded


def select_row_soft_constraints(row, require_verified=True, split=None):
    nested = row.get("soft_constraints", {}).get("constraints", [])
    raw = row.get("soft")
    raw = json.loads(raw) if isinstance(raw, str) else raw
    if raw is None:
        raw = nested
    if not isinstance(raw, list) or not all(isinstance(v, dict) for v in raw):
        raise ValueError("Soft constraints phải là list object JSON.")
    # Verification status belongs to the full dataset, not to the stripped field.
    provenance = {c.get("id"): c for c in nested if isinstance(c, dict)}
    merged = []
    for value in raw:
        item = dict(value)
        origin = provenance.get(item.get("id"), {})
        origin_status = origin.get("verification_status")
        if origin_status is not None and origin_status not in ACCEPTED_SOFT_STATUSES:
            item["verification_status"] = origin_status
        else:
            for key in ("verification_status", "verified_by", "category"):
                if key not in item and key in origin:
                    item[key] = origin[key]
        merged.append(item)
    return normalize_soft_constraints(merged, require_verified, split)

import json
import hashlib
from pathlib import Path
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if not tokenizer.chat_template:
    raise ValueError("Checkpoint không có chat template chính thức; kiểm tra lại MODEL_NAME.")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"
CHAT_TEMPLATE_KWARGS = {"enable_thinking": False}

def extract_text_from_completion(comp):
    if isinstance(comp, list):
        messages = [m for m in comp if isinstance(m, dict) and m.get("role") == "assistant"]
        comp = messages[-1] if messages else (comp[-1] if comp else "")
    text = comp.get("content", "") if isinstance(comp, dict) else comp
    text = str(text or "")
    if "</think>" in text:
        text = text.rsplit("</think>", 1)[-1]
    elif "<think>" in text:
        return ""
    return text.strip()

def prompt_messages(prompt):
    if isinstance(prompt, str) and prompt.strip():
        return [{"role": "user", "content": prompt}]
    if isinstance(prompt, list) and prompt and all(
        isinstance(m, dict) and m.get("role") in ("system", "user", "assistant") and isinstance(m.get("content"), str)
        for m in prompt
    ):
        return prompt
    raise ValueError("Prompt phải là chuỗi không rỗng hoặc danh sách chat message chứa content dạng text.")

def prompt_for_judge(prompt):
    messages = prompt_messages(prompt)
    return "\n\n".join(f"[{m['role']}]\n{m['content']}" for m in messages)

def parse_spec(spec, name):
    value = json.loads(spec) if isinstance(spec, str) else spec
    if not isinstance(value, list) or not all(isinstance(x, dict) for x in value):
        raise ValueError(f"{name} phải là list các object JSON.")
    return value

def normalize_row(row, split=None):
    row = dict(row)
    row["prompt"] = prompt_messages(row.get("prompt", ""))
    v_spec = row.get("verifier")
    if v_spec is None:
        v_spec = []
        for gt in row.get("hard_constraints", {}).get("ground_truth", []):
            ids = gt.get("instruction_id", gt.get("id"))
            kw = gt.get("kwargs", {})
            if isinstance(ids, list):
                kws = kw if isinstance(kw, list) else [kw] * len(ids)
                if len(kws) != len(ids):
                    raise ValueError("Số instruction_id và kwargs không khớp.")
                v_spec.extend({"id": i, "kwargs": k} for i, k in zip(ids, kws))
            else:
                v_spec.append({"id": ids, "kwargs": kw})
    hard = parse_spec(v_spec, "verifier")
    if not hard or any(not h.get("id") or not isinstance(h.get("kwargs", {}), dict) for h in hard):
        raise ValueError(f"Verifier thiếu hoặc không hợp lệ cho key={row.get('key')}.")
    if split and row.get('constraint_split') and row['constraint_split']!=split:
        raise ValueError(f"key={row.get('key')}: constraint_split không khớp file dữ liệu.")
    soft, excluded = select_row_soft_constraints(row, split=split or row.get('constraint_split'))
    if excluded or not soft:
        raise ValueError(f"key={row.get('key')}: rubric REJECT/UNVERIFIED hoặc thiếu provenance. "
                         "Upload ba dataset soft-v2 đã sửa; không dùng Input cũ bị strip verification_status.")
    row['reward_spec'] = soft_reward_spec()
    row["verifier"] = json.dumps(hard, ensure_ascii=False)
    row["soft"] = json.dumps(soft, ensure_ascii=False)
    row["num_hard"] = len(hard)
    return row

def load_rows(path, limit=0):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"Thiếu dataset: {path}. Đặt dataset thật trên Google Drive trước khi chạy.")
    content = path.read_text(encoding="utf-8-sig").strip()
    if not content or content.startswith("<"):
        raise ValueError(f"Dataset rỗng hoặc là HTML: {path}")
    rows = json.loads(content) if content.startswith("[") else [json.loads(l) for l in content.splitlines() if l.strip()]
    if not isinstance(rows, list) or not rows:
        raise ValueError(f"Dataset phải có ít nhất một mẫu: {path}")
    split = 'unseen' if path.name=='test_unseen.jsonl' or str(path)==globals().get('TEST_UNSEEN_FILE') else 'seen'
    rows = [normalize_row(r,split=split) for r in (rows[:limit] if limit > 0 else rows)]
    keys = [r.get("key") for r in rows]
    if any(not isinstance(k, str) or not k for k in keys) or len(keys) != len(set(keys)):
        raise ValueError(f"Dataset cần key không rỗng và duy nhất trong mỗi split: {path}")
    return rows

def baseline_key(split, row):
    # Seen và unseen có thể cùng key nhưng khác prompt; không tái sử dụng nhầm output.
    payload = json.dumps([MODEL_NAME, split, row["key"], row["prompt"], MAX_COMPLETION_LENGTH, CHAT_TEMPLATE_KWARGS], ensure_ascii=False, sort_keys=True)
    return hashlib.sha256(payload.encode()).hexdigest()

def save_json(path, value):
    # Ghi file tạm rồi thay thế, tránh file JSON dở dang nếu cell bị ngắt.
    path = Path(path)
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
    temp.replace(path)



def source_groups(all_rows):
    """Ghép các biến thể cùng key và cùng đề bài gốc; không tách theo ràng buộc."""
    parent={}
    def find(key):
        parent.setdefault(key,key)
        if parent[key]!=key:
            parent[key]=find(parent[key])
        return parent[key]
    def union(a,b):
        a,b=find(a),find(b)
        if a!=b:
            lo,hi=sorted((a,b))
            parent[hi]=lo
    for row in all_rows:
        key='key:'+row['key']
        find(key)
        base=row.get('base_instruction')
        if not base:
            user=[m['content'] for m in row['prompt'] if m['role']=='user'][-1]
            base=user.split('\n\nVui lòng tuân thủ nghiêm ngặt',1)[0]
        if isinstance(base,str) and base.strip():
            text=' '.join(base.split()).casefold()
            union(key,'base:'+hashlib.sha256(text.encode()).hexdigest())
    return {row['key']:find('key:'+row['key']) for row in all_rows}

def prepare_data_splits(train_rows,seen_rows,unseen_rows):
    import math
    groups=source_groups(train_rows+seen_rows+unseen_rows)
    train_groups={groups[r['key']] for r in train_rows}
    # Giữ ngoài train TẤT CẢ đề bài của test unseen, kể cả biến thể seen.
    test_groups={groups[r['key']] for r in unseen_rows}
    if test_groups==train_groups or not train_groups-test_groups:
        raise ValueError('Unseen dùng toàn bộ đề bài train; cần thêm đề bài riêng để tạo train/val/test độc lập.')
    def order(group):
        return hashlib.sha256(f'{SEED}:{group}'.encode()).hexdigest()
    target_test=max(1,math.ceil(len(train_groups)*TEST_FRACTION))
    more=max(0,target_test-len(train_groups & test_groups))
    available=sorted(train_groups-test_groups,key=order)
    test_groups.update(available[:more])
    available=sorted(train_groups-test_groups,key=order)
    val_count=max(1,math.ceil(len(train_groups)*VALIDATION_FRACTION))
    if len(available)<=val_count:
        raise ValueError('Không đủ đề bài để tách validation và train sau khi giữ test unseen.')
    val_groups=set(available[:val_count])
    fit_groups=set(available[val_count:])
    fit=[r for r in train_rows if groups[r['key']] in fit_groups]
    # Ưu tiên rubric của seen benchmark, fallback chính mẫu train được giữ lại.
    val_map={r['key']:r for r in train_rows if groups[r['key']] in val_groups}
    val_map.update({r['key']:r for r in seen_rows if groups[r['key']] in val_groups})
    val=sorted(val_map.values(),key=lambda r:order(r['key']))
    seen_test=[r for r in seen_rows if groups[r['key']] in test_groups]
    if not fit or not val or not seen_test or not unseen_rows:
        raise ValueError('Một split bị rỗng; kiểm tra quan hệ key/đề bài của ba file input.')
    manifest={
        'soft_policy_version':SOFT_POLICY_VERSION,'seed':SEED,'grouping':'same key OR normalized base instruction',
        'train_keys':[r['key'] for r in fit], 'validation_keys':[r['key'] for r in val],
        'test_seen_keys':[r['key'] for r in seen_test], 'test_unseen_keys':[r['key'] for r in unseen_rows],
        'train_groups':sorted(fit_groups),'validation_groups':sorted(val_groups),'test_groups':sorted(test_groups),
        'train_count':len(fit),'validation_count':len(val),'seen_test_count':len(seen_test),'unseen_test_count':len(unseen_rows),
        'dataset_sha256':hashlib.sha256(json.dumps([train_rows,seen_rows,unseen_rows],ensure_ascii=False,sort_keys=True).encode()).hexdigest(),
    }
    assert not (fit_groups & val_groups or fit_groups & test_groups or val_groups & test_groups)
    return fit,val,seen_test,list(unseen_rows),manifest

all_train_rows=load_rows(TRAIN_FILE)
all_seen_rows=load_rows(TEST_SEEN_FILE)
all_unseen_rows=load_rows(TEST_UNSEEN_FILE)
TRAIN_ROWS,VALIDATION_ROWS,TEST_SEEN_ROWS,TEST_UNSEEN_ROWS,SPLIT_MANIFEST=prepare_data_splits(all_train_rows,all_seen_rows,all_unseen_rows)
SPLIT_SIGNATURE=hashlib.sha256(json.dumps(SPLIT_MANIFEST,sort_keys=True).encode()).hexdigest()
if SPLIT_MANIFEST_IMPORT_FILE:
    imported_manifest=json.loads(Path(SPLIT_MANIFEST_IMPORT_FILE).read_text(encoding='utf-8'))
    if imported_manifest!=SPLIT_MANIFEST:
        raise ValueError('Split manifest Kaggle không khớp. Dùng đúng ba file dữ liệu gốc và cùng SEED/tỷ lệ split; không dùng file eval đã lọc làm dữ liệu gốc.')
save_json(os.path.join(RESULTS_DIR,'split_manifest.json'),SPLIT_MANIFEST)
VALIDATION_ROWS=VALIDATION_ROWS[:MAX_VALIDATION_SAMPLES]
if MAX_EVAL_SAMPLES_SEEN>0:
    TEST_SEEN_ROWS=TEST_SEEN_ROWS[:MAX_EVAL_SAMPLES_SEEN]
if MAX_EVAL_SAMPLES_UNSEEN>0:
    TEST_UNSEEN_ROWS=TEST_UNSEEN_ROWS[:MAX_EVAL_SAMPLES_UNSEEN]
TRAIN_PROMPTS={prompt_for_judge(r['prompt']) for r in TRAIN_ROWS}
print(f"Split: train={len(TRAIN_ROWS)}, validation pool={SPLIT_MANIFEST['validation_count']} (chọn {len(VALIDATION_ROWS)} mẫu cố định), test seen={len(TEST_SEEN_ROWS)}, test unseen={len(TEST_UNSEEN_ROWS)}")
print('Không dùng đề bài test trong train/validation; không dùng test chọn checkpoint.')
from collections import Counter
SOFT_DATA_QUALITY = {
    split: {'rows':len(rows),'criteria':sum(len(parse_spec(r['soft'],'soft')) for r in rows),
            'categories':dict(Counter(c['category'] for r in rows for c in parse_spec(r['soft'],'soft')))}
    for split,rows in [('train',TRAIN_ROWS),('validation',VALIDATION_ROWS),('seen',TEST_SEEN_ROWS),('unseen',TEST_UNSEEN_ROWS)]
}
for split,report in SOFT_DATA_QUALITY.items():
    expected=UNSEEN_SOFT_CATEGORIES if split=='unseen' else SEEN_SOFT_CATEGORIES
    report['missing_categories']=sorted(set(expected)-set(report['categories']))
save_json(os.path.join(RESULTS_DIR,'soft_data_quality.json'),SOFT_DATA_QUALITY)
print('Soft rubric coverage:',json.dumps(SOFT_DATA_QUALITY,ensure_ascii=False))


In [ ]:
# @title 4. Hàm model và cache baseline tương thích Kaggle
import gc
import torch
from transformers import GenerationConfig, Qwen3_5ForConditionalGeneration, set_seed
from tqdm.auto import tqdm

set_seed(SEED)
USE_BF16 = MODEL_DEVICE == "cuda" and torch.cuda.is_available() and torch.cuda.is_bf16_supported()
# T4/P100: giữ base weights FP32 và dùng autocast FP16. GPU hỗ trợ BF16: dùng BF16.
MODEL_DTYPE = torch.bfloat16 if USE_BF16 else torch.float32

def _qwen_text_position_pre_hook(module, args, kwargs):
    # Transformers 5.3.0 giữ rope_deltas của batch rollout (4 mẫu).
    # GRPO forward batch nhỏ hơn (1 mẫu) có thể repeat_interleave(1 // 4),
    # tạo batch vị trí rỗng. Truyền position_ids theo attention_mask của
    # CHÍNH batch hiện tại để không đi qua nhánh dùng rope_deltas cũ.
    # Hook chỉ sửa forward text chưa có position_ids; generate() đã tạo
    # position_ids và đường ảnh/video vẫn dùng xử lý chính thức của Qwen.
    if args or kwargs.get("position_ids") is not None:
        return args, kwargs
    if any(kwargs.get(k) is not None for k in (
        "pixel_values", "pixel_values_videos", "image_grid_thw", "video_grid_thw",
    )):
        return args, kwargs
    tokens = kwargs.get("input_ids")
    if tokens is None:
        tokens = kwargs.get("inputs_embeds")
    if tokens is None:
        return args, kwargs
    batch_size, seq_length = tokens.shape[:2]
    if batch_size == 0 or seq_length == 0:
        raise ValueError("Qwen forward nhận batch hoặc chuỗi token rỗng.")
    mask = kwargs.get("attention_mask")
    if mask is not None:
        if mask.ndim != 2 or mask.shape[0] != batch_size or mask.shape[1] < seq_length:
            raise ValueError("Text forward cần attention_mask 2D khớp batch và độ dài token.")
        positions = mask.long().cumsum(-1) - 1
        positions = positions.masked_fill(mask == 0, 1)[:, -seq_length:]
    else:
        cache = kwargs.get("past_key_values")
        past_length = cache.get_seq_length() if cache is not None else 0
        positions = torch.arange(past_length, past_length + seq_length,
                                 device=tokens.device).unsqueeze(0).expand(batch_size, -1)
    kwargs = dict(kwargs)
    kwargs["position_ids"] = positions.to(tokens.device)
    return args, kwargs


def align_qwen_special_tokens(qwen_model, tok):
    # Đồng bộ theo tokenizer thật, không hard-code token ID và không thêm token.
    for config in (qwen_model.config, qwen_model.config.get_text_config()):
        for name in ("pad_token_id", "bos_token_id", "eos_token_id"):
            setattr(config, name, getattr(tok, name))
    generation = qwen_model.generation_config
    generation.pad_token_id = tok.pad_token_id
    generation.bos_token_id = tok.bos_token_id
    # Giữ các EOS gốc của checkpoint, giống cách Trainer căn chỉnh EOS.
    existing_eos = generation.eos_token_id
    eos_ids = [] if existing_eos is None else (
        list(existing_eos) if isinstance(existing_eos, (list, tuple)) else [existing_eos]
    )
    if tok.eos_token_id is not None and tok.eos_token_id not in eos_ids:
        eos_ids.insert(0, tok.eos_token_id)
    generation.eos_token_id = eos_ids or None


def load_qwen_model(device=None, dtype=None):
    device = device or MODEL_DEVICE
    dtype = dtype or (torch.float32 if device == "cpu" else MODEL_DTYPE)
    # Giữ đúng lớp/checkpoint vision-language và tên trọng số LoRA.
    qwen_model = Qwen3_5ForConditionalGeneration.from_pretrained(
        MODEL_NAME, dtype=dtype,
        device_map={"": device}, attn_implementation="sdpa",
    )
    align_qwen_special_tokens(qwen_model, tokenizer)
    qwen_model.model.register_forward_pre_hook(_qwen_text_position_pre_hook, with_kwargs=True)
    return qwen_model


def generate_sample_response(eval_model, tok, prompt, max_new_tokens=None, return_metadata=False):
    eval_model.eval()
    formatted = tok.apply_chat_template(prompt_messages(prompt), tokenize=False,
        add_generation_prompt=True, **CHAT_TEMPLATE_KWARGS)
    inputs = tok(formatted, return_tensors="pt", add_special_tokens=False).to(eval_model.device)
    if inputs.input_ids.shape[1] > MAX_PROMPT_LENGTH:
        raise ValueError(f"Prompt dài {inputs.input_ids.shape[1]} token, vượt MAX_PROMPT_LENGTH={MAX_PROMPT_LENGTH}. Tăng cấu hình; không cắt mất ràng buộc.")
    with torch.inference_mode(), torch.autocast(
        device_type=eval_model.device.type,
        dtype=torch.bfloat16 if USE_BF16 else torch.float16,
        enabled=eval_model.device.type == "cuda",
    ):
        output = eval_model.generate(**inputs,
            max_new_tokens=max_new_tokens or MAX_COMPLETION_LENGTH,
            do_sample=False, pad_token_id=tok.pad_token_id, use_cache=True)
    generated=output[0][inputs.input_ids.shape[1]:]
    text=extract_text_from_completion(tok.decode(generated,skip_special_tokens=True))
    eos=eval_model.generation_config.eos_token_id
    eos_ids=eos if isinstance(eos,list) else [eos if eos is not None else tok.eos_token_id]
    hit_eos=generated.numel()>0 and generated[-1].item() in eos_ids
    result={'text':text,'generated_tokens':generated.numel(),
            'truncated':not hit_eos and generated.numel()>=(max_new_tokens or MAX_COMPLETION_LENGTH)}
    return result if return_metadata else text

BASELINE_FILE = os.path.join(RESULTS_DIR, "baseline_outputs.json")

def load_baseline_cache():
    # Nạp từ đĩa cả khi chỉ chạy lại title 8; không phụ thuộc dict từ lần train trước.
    paths=[Path(BASELINE_IMPORT_FILE)] if BASELINE_IMPORT_FILE else []
    paths.append(Path(BASELINE_FILE))
    cache={}
    for path in paths:
        if not path.is_file():
            if path == Path(BASELINE_FILE):
                continue
            raise FileNotFoundError(f'Thiếu baseline import: {path}')
        values=json.loads(path.read_text(encoding='utf-8'))
        if not isinstance(values,dict):
            raise ValueError(f'{path} phải là baseline cache JSON object, không phải CSV kết quả eval.')
        for key,value in values.items():
            if not isinstance(key,str) or not isinstance(value,dict) or not isinstance(value.get('text'),str) or \
               type(value.get('generated_tokens')) is not int or value['generated_tokens']<0 or \
               type(value.get('truncated')) is not bool:
                raise ValueError(f'Baseline cache sai định dạng tại key={key!r}: {path}. Dùng cache metadata của Kaggle/Colab mới, không phải cache Colab cũ chỉ chứa chuỗi.')
        cache.update(values)
    return cache

baseline_outputs=load_baseline_cache()
if BASELINE_IMPORT_FILE:
    save_json(BASELINE_FILE,baseline_outputs)

def ensure_baseline_outputs():
    global baseline_outputs
    baseline_outputs=load_baseline_cache()
    required=[(split,row) for split,rows in (
        ('validation',VALIDATION_ROWS),('seen',TEST_SEEN_ROWS),('unseen',TEST_UNSEEN_ROWS))
        for row in rows]
    missing=[(split,row) for split,row in required if baseline_key(split,row) not in baseline_outputs]
    if not EVALUATE_BENCHMARK_BASELINE:
        print('Baseline benchmark tắt; validation step 0 sẽ sinh bằng base model.')
        return
    if not missing:
        print(f'Baseline cache đầy đủ ({len(required)} mẫu); bỏ qua nạp model và inference ở title 4.')
        return
    model=load_qwen_model()
    try:
        for split,row in tqdm(missing,desc='Baseline còn thiếu'):
            cache_id=baseline_key(split,row)
            baseline_outputs[cache_id]=generate_sample_response(model,tokenizer,row['prompt'],return_metadata=True)
            save_json(BASELINE_FILE,baseline_outputs)
        print(f'Baseline đã lưu: {BASELINE_FILE}')
    finally:
        del model
        gc.collect()
        torch.cuda.empty_cache()

ensure_baseline_outputs()
